# Conflict-trained 2-unit GRU — v8

Trains a 2-unit GRU that is taught **auditory dominance** on conflict trials, so its dynamics can be compared
with the standard network. Only **one thing** changes: in the standard setup the localisation-conflict trials
are trained with random labels, and here they are relabelled so the **auditory side always wins**. Everything
else is identical (same trials, same X, same other labels, same sizes, seeds, epochs, optimiser), so any
difference in dynamics is attributable to the conflict training alone.

Relabelling: `loc_conflict_audL_visR` (audio on the left) -> label 3 (left);
`loc_conflict_audR_visL` (audio on the right) -> label 2 (right). `det_multisensory` stays held out.

Loads data from `./generated_trials_v8`.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.optimize import minimize
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
DATA_DIR = Path("./generated_trials_v8"); OUT_DIR = Path("./conflict_trained_v8"); OUT_DIR.mkdir(exist_ok=True)
HIDDEN = 2; T_ON, T_OFF = 10, 20
CLASS_NAMES  = ["no det (0)", "det (1)", "right (2)", "left (3)"]
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]

## 2. Load data and apply the single change (relabel conflict trials to audio-wins)

In [ ]:
def load(split):
    d = np.load(DATA_DIR / f"{split}.npz", allow_pickle=True)
    out = {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64), "types": d["types"]}
    if "aud_int" in d:
        out["aud_int"] = d["aud_int"].astype(np.float32); out["vis_int"] = d["vis_int"].astype(np.float32)
    return out
train, test = load("train"), load("test")
T = test["X"].shape[2]
SUBTASKS = sorted(set(test["types"]))
CONFLICT = ["det_multisensory", "loc_conflict_audL_visR", "loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
AUDIO_WINS = {"loc_conflict_audL_visR": 3, "loc_conflict_audR_visL": 2}   # audio side label

# The ONE change: relabel loc-conflict training trials to audio-wins. Everything else untouched.
y_train = train["y"].copy()
n_changed = 0
for s, lab in AUDIO_WINS.items():
    m = train["types"] == s
    y_train[m] = lab; n_changed += int(m.sum())
print("Relabelled %d conflict training trials to audio-wins. X and all other labels unchanged." % n_changed)
print("Class counts now:", np.bincount(y_train, minlength=4))

## 3. Model (identical to the standard unified network)

In [ ]:
class UnifiedGRU(nn.Module):
    def __init__(self, n_channels=4, hidden_size=2, n_classes=4):
        super().__init__()
        self.gru = nn.GRU(n_channels, hidden_size, batch_first=True)
        self.readout = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        x = x.transpose(1, 2); h, _ = self.gru(x); return self.readout(h)

## 4. Train (all seeds kept; no quality selection)

In [ ]:
def train_model(seed, y_labels, n_epochs=50, lr=1e-3, batch=64):
    torch.manual_seed(seed); np.random.seed(seed)
    model = UnifiedGRU(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train["X"]), torch.from_numpy(y_labels)),
                        batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            logits = model(Xb); B, Tt, C = logits.shape
            loss = loss_fn(logits.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model):
    model.eval()
    with torch.no_grad(): pred = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).numpy()
    accs = {s: float((pred[test["types"] == s] == test["y"][test["types"] == s]).mean()) for s in SUBTASKS}
    return pred, accs, np.mean([accs[s] for s in NONCONF])

def auditory_dominance(pred):
    """Fraction of localisation-conflict test trials where the network followed the audio side."""
    out = {}
    for s, lab in AUDIO_WINS.items():
        m = test["types"] == s
        out[s] = float((pred[m] == lab).mean())
    return out

SEEDS = [0, 1, 2, 3, 4]
models = {}
for s in SEEDS:
    m = train_model(s, y_train); models[s] = m
    pred, _, nc = evaluate(m); ad = auditory_dominance(pred)
    print("seed %d  non-conflict %.3f   audio-follow: audL_visR %.2f  audR_visL %.2f" %
          (s, nc, ad["loc_conflict_audL_visR"], ad["loc_conflict_audR_visL"]))
DISPLAY_SEED = SEEDS[0]          # seed shown below; set to any value in SEEDS
model = models[DISPLAY_SEED]
pred, accs, _ = evaluate(model)
print("\nseed %d per-subtask:" % DISPLAY_SEED)
for s in SUBTASKS: print("  %-26s %.3f" % (s, accs[s]))

## 5. Hand-coded update, verified against PyTorch

In [ ]:
def gru_params(model):
    g = model.gru
    Wih = g.weight_ih_l0.detach().numpy(); Whh = g.weight_hh_l0.detach().numpy()
    bih = g.bias_ih_l0.detach().numpy();   bhh = g.bias_hh_l0.detach().numpy()
    H = Whh.shape[1]; sl = lambda M, i: M[i*H:(i+1)*H]
    return dict(Wir=sl(Wih,0), Wiz=sl(Wih,1), Win=sl(Wih,2), Whr=sl(Whh,0), Whz=sl(Whh,1), Whn=sl(Whh,2),
                bir=bih[:H], biz=bih[H:2*H], bin_=bih[2*H:3*H], bhr=bhh[:H], bhz=bhh[H:2*H], bhn=bhh[2*H:3*H], H=H)
def sigmoid(v): return 1.0/(1.0+np.exp(-v))
def gru_step(h, x, p):
    r = sigmoid(p["Wir"]@x + p["bir"] + p["Whr"]@h + p["bhr"])
    z = sigmoid(p["Wiz"]@x + p["biz"] + p["Whz"]@h + p["bhz"])
    n = np.tanh(p["Win"]@x + p["bin_"] + r*(p["Whn"]@h + p["bhn"]))
    return (1.0-z)*n + z*h
P = gru_params(model); Wro = model.readout.weight.detach().numpy(); bro = model.readout.bias.detach().numpy()
def readout_class(h): return int(np.argmax(Wro@h + bro))
with torch.no_grad(): h_pt,_ = model.gru(torch.from_numpy(test["X"][:6]).transpose(1,2)); h_pt=h_pt.numpy()
err=0.0
for b in range(6):
    h=np.zeros(HIDDEN)
    for t in range(T): h=gru_step(h,test["X"][b][:,t],P); err=max(err,abs(h-h_pt[b,t]).max())
print("max |numpy - torch| =", err); assert err<1e-4; print("OK")

## 6. Dynamics: decision regions, trajectories, fixed points

Same analysis as the standard 2-unit notebook. The signature of auditory dominance is the conflict trajectories collapsing onto the audio-side region rather than splitting.

In [ ]:
GRID=np.linspace(-1.05,1.05,350); GX,GY=np.meshgrid(GRID,GRID); _flat=np.stack([GX.ravel(),GY.ravel()],1)
def draw_regions(ax, fill=True):
    reg=np.argmax(_flat@Wro.T+bro,1).reshape(GX.shape)
    if fill: ax.pcolormesh(GX,GY,reg,cmap=ListedColormap(CLASS_COLORS),alpha=0.20,shading="auto",vmin=0,vmax=3)
    ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2"); ax.set_aspect("equal")
    ax.set_xlim(-1.05,1.05); ax.set_ylim(-1.05,1.05)
def label_regions(ax, fs=11):
    reg=np.argmax(_flat@Wro.T+bro,1)
    for cls in range(4):
        m=reg==cls
        if m.sum()<50: continue
        ax.text(_flat[m,0].mean(),_flat[m,1].mean(),CLASS_NAMES[cls],ha="center",va="center",fontsize=fs,fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.25",fc="white",ec=CLASS_COLORS[cls],lw=1.5,alpha=0.85),zorder=20)
def hidden_traj(idx):
    out=np.zeros((len(idx),T+1,HIDDEN))
    for k,i in enumerate(idx):
        h=np.zeros(HIDDEN)
        for t in range(T): h=gru_step(h,test["X"][i][:,t],P); out[k,t+1]=h
    return out
fig,ax=plt.subplots(figsize=(8.5,8)); draw_regions(ax); label_regions(ax)
pred_all=model(torch.from_numpy(test["X"]))[:,-1,:].argmax(-1).detach().numpy()
cols=plt.cm.tab20(np.linspace(0,1,20)); ci=0
for s in NONCONF:
    tr=hidden_traj(np.where(test["types"]==s)[0]).mean(0); c=cols[ci]; ci+=1
    ax.plot(tr[:,0],tr[:,1],color=c,lw=2,label=s); ax.scatter(*tr[-1],color=c,s=40,marker="*",edgecolor="k",lw=0.4,zorder=6)
for s in CONFLICT:
    labs=[("det",1),("no-det",0)] if s=="det_multisensory" else [("right",2),("left",3)]
    for nm,lb in labs:
        idx=np.where((test["types"]==s)&(pred_all==lb))[0]
        if len(idx)==0: continue
        tr=hidden_traj(idx).mean(0); c=cols[ci%20]; ci+=1
        ax.plot(tr[:,0],tr[:,1],color=c,lw=2.4,ls="--",label="%s->%s"%(s,nm)); ax.scatter(*tr[-1],color=c,s=55,marker="*",edgecolor="k",lw=0.5,zorder=6)
ax.scatter(0,0,color="k",s=30,zorder=7)
ax.set_title("Conflict-trained GRU: average trajectory per subtask (seed %d)"%DISPLAY_SEED)
ax.legend(loc="center left",bbox_to_anchor=(1.01,0.5),fontsize=7.5,frameon=False)
plt.tight_layout(); plt.savefig(OUT_DIR/"conflict_fig1.png",dpi=150,bbox_inches="tight"); plt.show()

## 7. Baseline fixed points and flow field

In [ ]:
def fixed_points(x,p,inits,tol=1e-10):
    def q(h): d=gru_step(h,x,p)-h; return 0.5*float(d@d)
    found=[]
    for h0 in inits:
        r=minimize(q,h0,method="L-BFGS-B",bounds=[(-1.2,1.2)]*p["H"],options=dict(maxiter=500))
        if r.fun<tol: found.append(r.x)
    uniq=[]
    for h in found:
        if not any(np.allclose(h,u,atol=1e-3) for u in uniq): uniq.append(h)
    return uniq
def jacobian(h,x,p,eps=1e-5):
    H=p["H"]; J=np.zeros((H,H)); f0=gru_step(h,x,p)
    for i in range(H):
        hp=h.copy(); hp[i]+=eps; J[:,i]=(gru_step(hp,x,p)-f0)/eps
    return J
def classify(h,x,p):
    ev=np.linalg.eigvals(jacobian(h,x,p)); m=np.abs(ev)
    return ("stable" if np.all(m<1) else "unstable" if np.all(m>1) else "saddle"), ev
def flow(ax,x,p,n=23,scale=8):
    g=np.linspace(-1,1,n); XX,YY=np.meshgrid(g,g); U=np.zeros_like(XX); V=np.zeros_like(YY)
    for i in range(n):
        for j in range(n):
            d=gru_step(np.array([XX[i,j],YY[i,j]]),x,p)-np.array([XX[i,j],YY[i,j]]); U[i,j],V[i,j]=d
    ax.quiver(XX,YY,U,V,color="0.35",alpha=0.6,scale=scale,width=0.003)
ends=[]
for i in range(0,len(test["X"]),7):
    h=np.zeros(HIDDEN)
    for t in range(T): h=gru_step(h,test["X"][i][:,t],P)
    ends.append(h)
inits=ends+[np.random.uniform(-1,1,HIDDEN) for _ in range(150)]
x0=np.zeros(4); fps=fixed_points(x0,P,inits)
fig,ax=plt.subplots(figsize=(7,7)); draw_regions(ax); label_regions(ax); flow(ax,x0,P)
print("Baseline fixed points (conflict-trained):")
for h in fps:
    k,ev=classify(h,x0,P); mk={"stable":"o","saddle":"X","unstable":"^"}[k]
    ax.scatter(*h,marker=mk,s=130,edgecolor="k",facecolor="white",linewidths=1.6,zorder=6)
    print("  h=[% .3f % .3f] %-8s -> %-10s |eig|=%s"%(h[0],h[1],k,CLASS_NAMES[readout_class(h)],np.round(np.abs(ev),3)))
ax.set_title("Conflict-trained GRU: baseline flow field and fixed points (seed %d)"%DISPLAY_SEED)
plt.tight_layout(); plt.savefig(OUT_DIR/"conflict_flow.png",dpi=150,bbox_inches="tight"); plt.show()

## 8. Notes

- The only change from the standard network is the conflict-trial labelling (random -> audio-wins). Cell 2
  prints how many trials changed; everything else is identical.
- The audio-follow rates in cell 4 quantify how strongly each network took on auditory dominance.
- The comparison with the standard 2-unit network is at the same `DISPLAY_SEED`: audio-dominant training
  should pull the conflict trajectories onto the audio-side region, echoing the Song et al. mouse result.